# Chapter 9: Verifiers — Routing Claims to GMS Primitives

**Learning objectives:**
- Understand the verifier registry and routing architecture
- Run individual verifiers (Graph, Numeric, Artifact, Logic, Policy) to produce SubVerdict objects
- Use the VerificationRouter to dispatch claims automatically

---
## Preamble

Chapter 8 introduced the claim layer — six typed claims that decompose agent output into verifiable units. But claims are inert data structures; they carry assertions, not answers. This chapter introduces **verifiers** — the modules that take each claim and call the appropriate GMS primitive to produce a **SubVerdict** with a pass/fail decision, confidence score, raw geometric signal and structured failure attribution.

The architecture follows a single design principle: **GMS is the authority.** Every verification decision routes through a GMS method. If GMS cannot verify a claim (missing phase encoder, entity not in graph, relation not trained), the verifier fails rather than silently passing. No Python arithmetic, no tolerance heuristics, no bypasses.

Key modules:
- `knowlytix.harness.testing.verifiers` — `SubVerdict`, `GraphVerifier`, `NumericVerifier`, `ArtifactVerifier`, `LogicVerifier`, `PolicyVerifier`, `VerificationRouter`
- `knowlytix.harness.testing.failures` — `FailureCode`, `FailureAttribution`, `FailureSeverity`
- `knowlytix.harness.testing.claims` — claim types from Chapter 8

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the Chapter 4 store
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"\nStore: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

In [ ]:
# Calibrate the Chapter 6 judge and surface the three
# decision thresholds that Chapter 9 verifiers will consume.
from knowlytix.harness.testing import GMSJudge

judge = GMSJudge(store)
judge.calibrate(seed=42)

tau_geo = judge._thresholds.get("geodesic")
tau_hol = judge._thresholds.get("holonomy")
print("Calibrated thresholds from GMSJudge:")
print(f"  tau_geodesic (GraphVerifier.plausibility_threshold) = {tau_geo:.4f}")
print(f"  tau_holonomy (LogicVerifier.config.tau_path)         = {tau_hol:.4f}")
print("  PolicyVerifier / tau_ent / tau_contra: no calibration corpus in ch04 store -> defaults retained")

---

## The SubVerdict Data Model

Every verifier produces a `SubVerdict` — a structured record of the verification outcome. Its key fields are:

| Field | Type | Description |
|---|---|---|
| `verifier_name` | str | Which verifier produced this (e.g. "GraphVerifier") |
| `claim_id` | str | The claim that was verified |
| `passed` | bool | Pass/fail decision |
| `confidence` | float | Verification confidence in [0, 1] |
| `gms_method_used` | str | Which GMS primitive was called |
| `raw_score` | float | The raw geometric signal (geodesic, tension, defect, margin) |
| `detail` | str | Human-readable explanation |
| `failure` | FailureAttribution | Structured failure record (if failed) |

In [ ]:
# Import verifiers and claim types
from knowlytix.harness.testing.verifiers import (
    GraphVerifier,
    NumericVerifier,
    ArtifactVerifier,
    LogicVerifier,
    PolicyVerifier,
    VerificationRouter,
)
from knowlytix.harness.testing.claims import (
    ClaimProvenance,
    FactClaim, ThresholdClaim, RuleClaim,
    ArtifactClaim, ToolUseClaim, ClaimCompiler,
)

# Show the verifier registry
verifier_map = {
    "fact": ("GraphVerifier", "score_triple()", "geodesic < 1.2"),
    "threshold": ("NumericVerifier", "check_threshold()", "phase-encoded"),
    "artifact": ("ArtifactVerifier", "enm.read_exact()", "SHA-256 hash"),
    "rule": ("LogicVerifier", "check_holonomy()", "defect < tau_path"),
    "tool_use": ("PolicyVerifier", "score_triple()", "geodesic < 1.5"),
    "semantic": ("SemanticVerifier", "LLM judge + GMS", "score >= 3.0"),
}

print(f"{'Claim Type':<14s} {'Verifier':<20s} {'GMS Method':<22s} {'Decision Rule'}")
print("-" * 78)
for ctype, (vname, method, rule) in verifier_map.items():
    print(f"{ctype:<14s} {vname:<20s} {method:<22s} {rule}")

---

## Lab 9.1 — Verifying Claims Against GMS

We construct claims from Chapter 8 and run each through its designated verifier. This demonstrates the full verification cascade for each claim type.

### 9.1a GraphVerifier: FactClaims via `score_triple()`

The `GraphVerifier` uses a three-step cascade:
1. **Neural scoring** via `score_triple()` — geodesic distance on $S^{m-1}$ (if relation is trained)
2. **Graph lookup** via `query_triples(h, r, t)` — exact match in document graph
3. **Pair existence** via `query_triples(h, *, t)` — any relation between the entities

A triple passes if its geodesic distance < 1.2 (the plausibility threshold). Confidence = $e^{-\text{geodesic}}$.

In [5]:
# Lab 9.1a: GraphVerifier on FactClaims
# Use the calibrated geodesic threshold from the Chapter 6 judge.
graph_v = GraphVerifier(store, plausibility_threshold=tau_geo)
print(f"GraphVerifier.plausibility_threshold = {graph_v.plausibility_threshold:.4f} (calibrated)\n")

# Claim 1: A correct fact from the document
fact_correct = FactClaim(
    head="MDL-001",
    relation="has_risk_rating",
    tail="High",
    raw_text="MDL-001 has risk rating High",
    confidence=1.0,
    provenance=ClaimProvenance(extraction_method="metadata"),
)

sv1 = graph_v.verify(fact_correct)
print("=== FactClaim (correct triple) ===")
print(f"  Claim:      ({fact_correct.head}, {fact_correct.relation}, {fact_correct.tail})")
print(f"  Passed:     {sv1.passed}")
print(f"  Confidence: {sv1.confidence:.4f}")
print(f"  GMS method: {sv1.gms_method_used}")
print(f"  Raw score:  {sv1.raw_score}")
print(f"  Detail:     {sv1.detail}")

GraphVerifier.plausibility_threshold = 1.3883 (calibrated)

=== FactClaim (correct triple) ===
  Claim:      (MDL-001, has_risk_rating, High)
  Passed:     True
  Confidence: 0.2687
  GMS method: score_triple
  Raw score:  1.3142540454864502
  Detail:     (MDL-001, has_risk_rating, High) geodesic=1.3143


In [6]:
# Claim 2: An incorrect fact (wrong tail entity)
fact_wrong = FactClaim(
    head="MDL-001",
    relation="has_risk_rating",
    tail="Low",
    raw_text="MDL-001 has risk rating Low",
    confidence=1.0,
    provenance=ClaimProvenance(extraction_method="metadata"),
)

sv2 = graph_v.verify(fact_wrong)
print("=== FactClaim (incorrect triple) ===")
print(f"  Claim:      ({fact_wrong.head}, {fact_wrong.relation}, {fact_wrong.tail})")
print(f"  Passed:     {sv2.passed}")
print(f"  Confidence: {sv2.confidence:.4f}")
print(f"  GMS method: {sv2.gms_method_used}")
print(f"  Raw score:  {sv2.raw_score}")
print(f"  Detail:     {sv2.detail}")
if sv2.failure:
    print(f"  Failure:    {sv2.failure.code.value}")
    print(f"  Fail detail:{sv2.failure.detail}")

=== FactClaim (incorrect triple) ===
  Claim:      (MDL-001, has_risk_rating, Low)
  Passed:     False
  Confidence: 0.2230
  GMS method: score_triple
  Raw score:  1.500527024269104
  Detail:     (MDL-001, has_risk_rating, Low) geodesic=1.5005
  Failure:    fact_incorrect
  Fail detail:Triple (MDL-001, has_risk_rating, Low) geodesic=1.5005 > threshold=1.3883203653991223


In [7]:
# Claim 3: Entity not in the GMS store
fact_missing = FactClaim(
    head="MDL-999",
    relation="has_risk_rating",
    tail="High",
    raw_text="MDL-999 has risk rating High",
    confidence=1.0,
    provenance=ClaimProvenance(extraction_method="text_match"),
)

sv3 = graph_v.verify(fact_missing)
print("=== FactClaim (entity not found) ===")
print(f"  Claim:      ({fact_missing.head}, {fact_missing.relation}, {fact_missing.tail})")
print(f"  Passed:     {sv3.passed}")
print(f"  GMS method: {sv3.gms_method_used}")
print(f"  Detail:     {sv3.detail}")
if sv3.failure:
    print(f"  Failure:    {sv3.failure.code.value}")
    print(f"  GMS method: {sv3.failure.gms_method}")

=== FactClaim (entity not found) ===
  Claim:      (MDL-999, has_risk_rating, High)
  Passed:     False
  GMS method: fuzzy_match_entity
  Detail:     Head entity 'MDL-999' not found in GMS
  Failure:    entity_not_found
  GMS method: fuzzy_match_entity


In [8]:
# Summary: three FactClaim scenarios
print(f"\n{'Scenario':<25s} {'Passed':>7s} {'Confidence':>11s} {'GMS Method':<18s} {'Failure Code'}")
print("-" * 85)
for label, sv in [("Correct triple", sv1), ("Wrong tail", sv2), ("Missing entity", sv3)]:
    fc = sv.failure.code.value if sv.failure else "—"
    raw = f"{sv.raw_score:.4f}" if sv.raw_score is not None else "—"
    print(f"{label:<25s} {str(sv.passed):>7s} {sv.confidence:>11.4f} "
          f"{sv.gms_method_used:<18s} {fc}")


Scenario                   Passed  Confidence GMS Method         Failure Code
-------------------------------------------------------------------------------------
Correct triple               True      0.2687 score_triple       —
Wrong tail                  False      0.2230 score_triple       fact_incorrect
Missing entity              False      0.0000 fuzzy_match_entity entity_not_found


### NumericVerifier: ThresholdClaims via Phase Encoding

The `NumericVerifier` uses GMS phase-encoded inequality checks. It calls `lookup_enm()` for the stored value and `check_threshold()` for the comparison. If no phase encoder exists for the metric, it **fails** rather than falling back to Python arithmetic — GMS is the authority.

In [9]:
# Lab 9.1b: NumericVerifier
numeric_v = NumericVerifier(store)

# Find a real ENM entry to construct a valid threshold claim
enm_keys = list(store.enm.keys())
sample_key = enm_keys[0]
sample_val = float(store.enm.read_exact(sample_key).item())
print(f"Sample ENM entry: ({sample_key.type}, {sample_key.id}) = {sample_val}")

# Threshold claim that should PASS (value >= a lower limit)
threshold_pass = ThresholdClaim(
    entity=sample_key.id,
    metric=sample_key.type,
    value=sample_val,
    limit=sample_val - 0.01,  # slightly below actual
    operator="ge",
    confidence=1.0,
    provenance=ClaimProvenance(extraction_method="metadata"),
)

sv_tpass = numeric_v.verify(threshold_pass)
print("\n=== ThresholdClaim (should pass) ===")
print(f"  Claim:   {threshold_pass.entity}.{threshold_pass.metric} = "
      f"{threshold_pass.value} >= {threshold_pass.limit}")
print(f"  Passed:  {sv_tpass.passed}")
print(f"  Confidence: {sv_tpass.confidence}")
print(f"  GMS method: {sv_tpass.gms_method_used}")
print(f"  Detail:  {sv_tpass.detail}")

Sample ENM entry: (model_inventory_summary, MDL-001/CreditScore-XGB/Credit Decisioning/High/Degraded) = 1.0

=== ThresholdClaim (should pass) ===
  Claim:   MDL-001/CreditScore-XGB/Credit Decisioning/High/Degraded.model_inventory_summary = 1.0 >= 0.99
  Passed:  False
  Confidence: 0.0
  GMS method: check_threshold
  Detail:  No GMS phase encoder for metric 'model_inventory_summary' — cannot verify 1.0 ge 0.99


In [10]:
# Threshold claim that should FAIL (value >= a higher limit)
threshold_fail = ThresholdClaim(
    entity=sample_key.id,
    metric=sample_key.type,
    value=sample_val,
    limit=sample_val + 10.0,  # well above actual
    operator="ge",
    confidence=1.0,
    provenance=ClaimProvenance(extraction_method="metadata"),
)

sv_tfail = numeric_v.verify(threshold_fail)
print("=== ThresholdClaim (should fail) ===")
print(f"  Claim:   {threshold_fail.entity}.{threshold_fail.metric} = "
      f"{threshold_fail.value} >= {threshold_fail.limit}")
print(f"  Passed:  {sv_tfail.passed}")
print(f"  Confidence: {sv_tfail.confidence}")
print(f"  GMS method: {sv_tfail.gms_method_used}")
print(f"  Detail:  {sv_tfail.detail}")
if sv_tfail.failure:
    print(f"  Failure: {sv_tfail.failure.code.value}")
    print(f"  Margin:  {sv_tfail.failure.raw_score}")

=== ThresholdClaim (should fail) ===
  Claim:   MDL-001/CreditScore-XGB/Credit Decisioning/High/Degraded.model_inventory_summary = 1.0 >= 11.0
  Passed:  False
  Confidence: 0.0
  GMS method: check_threshold
  Detail:  No GMS phase encoder for metric 'model_inventory_summary' — cannot verify 1.0 ge 11.0
  Failure: verification_error
  Margin:  None


### ArtifactVerifier: ArtifactClaims via SHA-256 Hash

The `ArtifactVerifier` uses a three-step process:
1. Find the ENM key by `(category, entity_id)`
2. Verify ENM entry integrity via `enm.verify_integrity()` (recomputes SHA-256)
3. Hash-based comparison: serialize claimed value to IEEE 754 float64 bytes, compute SHA-256, compare to stored hash

This is deterministic and eliminates ambiguity about floating-point tolerance.

In [11]:
# Lab 9.1c: ArtifactVerifier — exact match
artifact_v = ArtifactVerifier(store)

# Correct claimed value (matches ENM exactly)
artifact_correct = ArtifactClaim(
    category=sample_key.type,
    entity_id=sample_key.id,
    claimed_value=sample_val,
    confidence=1.0,
    provenance=ClaimProvenance(extraction_method="numeric_match"),
)

sv_acorr = artifact_v.verify(artifact_correct)
print("=== ArtifactClaim (exact match) ===")
print(f"  Claim:   {artifact_correct.category}({artifact_correct.entity_id}) "
      f"= {artifact_correct.claimed_value}")
print(f"  Passed:  {sv_acorr.passed}")
print(f"  Confidence: {sv_acorr.confidence}")
print(f"  GMS method: {sv_acorr.gms_method_used}")
print(f"  Detail:  {sv_acorr.detail}")

=== ArtifactClaim (exact match) ===
  Claim:   model_inventory_summary(MDL-001/CreditScore-XGB/Credit Decisioning/High/Degraded) = 1.0
  Passed:  True
  Confidence: 1.0
  GMS method: enm.verify_integrity
  Detail:  ENM (model_inventory_summary, MDL-001/CreditScore-XGB/Credit Decisioning/High/Degraded): claimed=1.0, stored=1.0, SHA-256 match


In [12]:
# Wrong claimed value (off by 0.001 — SHA-256 catches it)
artifact_wrong = ArtifactClaim(
    category=sample_key.type,
    entity_id=sample_key.id,
    claimed_value=sample_val + 0.001,
    confidence=1.0,
    provenance=ClaimProvenance(extraction_method="numeric_match"),
)

sv_awrong = artifact_v.verify(artifact_wrong)
print("=== ArtifactClaim (off by 0.001) ===")
print(f"  Claim:   {artifact_wrong.category}({artifact_wrong.entity_id}) "
      f"= {artifact_wrong.claimed_value}")
print(f"  Passed:  {sv_awrong.passed}")
print(f"  Confidence: {sv_awrong.confidence}")
print(f"  GMS method: {sv_awrong.gms_method_used}")
print(f"  Detail:  {sv_awrong.detail}")
if sv_awrong.failure:
    print(f"  Failure: {sv_awrong.failure.code.value}")
    print(f"  Detail:  {sv_awrong.failure.detail}")

=== ArtifactClaim (off by 0.001) ===
  Claim:   model_inventory_summary(MDL-001/CreditScore-XGB/Credit Decisioning/High/Degraded) = 1.001
  Passed:  False
  Confidence: 0.0
  GMS method: enm.verify_integrity
  Detail:  ENM (model_inventory_summary, MDL-001/CreditScore-XGB/Credit Decisioning/High/Degraded): claimed=1.001, stored=1.0, SHA-256 MISMATCH
  Failure: exact_mismatch
  Detail:  SHA-256 mismatch: claimed=1.001 (hash=a83fef9115500b5f...) vs stored=1.0 (hash=6c3c396ed6b5c36d...)


### LogicVerifier: RuleClaims via Holonomy and Tension

The `LogicVerifier` handles two rule types:
- **Consistency**: checks path closure via `check_holonomy()` (rotor composition defect on $S^{m-1}$)
- **Contradiction**: checks tension energy between entity pairs via `tension_energy()` ($2\sin(\theta/2)$)

The verification cascade: holonomy -> graph hops -> tension energy between endpoints.

In [ ]:
# Lab 9.1d: LogicVerifier
# Route calibrated tau_path through the LogicVerifier config.
from knowlytix.knowledge.config import VerifyConfig
logic_cfg = VerifyConfig()
if tau_hol is not None:
    logic_cfg.tau_path = float(tau_hol)
logic_v = LogicVerifier(store, logic_cfg)
print(f"LogicVerifier.config.tau_path = {logic_cfg.tau_path:.4f} (calibrated)\n")

# Find two entities that share relations in the graph for a consistency check
triples = store.doc_graph.triples[:20]
print("Sample triples from document graph:")
for h, r, t in triples[:5]:
    print(f"  ({h}, {r}, {t})")

# Build a consistency RuleClaim from a 2-hop path in the graph
# Find: A --r1--> B --r2--> C
hop1 = triples[0]  # first triple
h1, r1, mid = hop1

# Find a second hop from mid
hop2 = None
r2, tail = None, None
for h, r, t in triples:
    if h == mid and t != h1:
        hop2 = (h, r, t)
        r2, tail = r, t
        break

if hop2:
    print(f"\n2-hop path: {h1} --{r1}--> {mid} --{r2}--> {tail}")

    # Find direct relation from h1 to tail (if any)
    direct_hits = store.query_triples(head=h1, tail=tail)
    direct_rel = direct_hits[0][1] if direct_hits else r1  # fallback

    rule_consist = RuleClaim(
        antecedent_entities=[h1, mid, tail],
        antecedent_relations=[r1, r2],
        consequent_entity=tail,
        direct_relation=direct_rel,
        rule_type="consistency",
        confidence=1.0,
        provenance=ClaimProvenance(extraction_method="metadata"),
    )

    sv_rule = logic_v.verify(rule_consist)
    print("\n=== RuleClaim (consistency) ===")
    print(f"  Path:     {h1} -> {mid} -> {tail}")
    print(f"  Rels:     [{r1}, {r2}]")
    print(f"  Direct:   {direct_rel}")
    print(f"  Passed:   {sv_rule.passed}")
    print(f"  Confidence: {sv_rule.confidence:.4f}")
    print(f"  GMS method: {sv_rule.gms_method_used}")
    print(f"  Raw score:{sv_rule.raw_score}")
    print(f"  Detail:   {sv_rule.detail[:120]}")
else:
    print("Could not find a 2-hop path (unexpected)")

In [14]:
# Contradiction check: two entities that should disagree
rule_contra = RuleClaim(
    antecedent_entities=[h1, mid],
    antecedent_relations=[],
    rule_type="contradiction",
    confidence=1.0,
    provenance=ClaimProvenance(extraction_method="metadata"),
)

sv_contra = logic_v.verify(rule_contra)
print("=== RuleClaim (contradiction check) ===")
print(f"  Entities: [{h1}, {mid}]")
print(f"  Passed:   {sv_contra.passed}")
print(f"  Confidence: {sv_contra.confidence:.4f}")
print(f"  GMS method: {sv_contra.gms_method_used}")
print(f"  Raw score:{sv_contra.raw_score}")
print(f"  Detail:   {sv_contra.detail[:120]}")
if sv_contra.failure:
    print(f"  Failure:  {sv_contra.failure.code.value}")

=== RuleClaim (contradiction check) ===
  Entities: [Organization, Meridian Financial Technologies, Inc.]
  Passed:   False
  Confidence: 0.2916
  GMS method: tension_energy
  Raw score:1.4167555570602417
  Detail:   Tension(Organization, Meridian Financial Technologies, Inc.)=1.4168 [neutral]
  Failure:  tension_exceeded


### PolicyVerifier: ToolUseClaims via Geometric Plausibility

The `PolicyVerifier` checks whether an agent's tool call is plausible within the GMS knowledge graph. It attempts to match the tool and result entities, then scores the triple `(tool, "produces", result)` via geodesic distance (threshold: 1.5). If neither entity is in the graph, it fails with `POLICY_VIOLATION`.

In [15]:
# Lab 9.1e: PolicyVerifier
policy_v = PolicyVerifier(store)

# ToolUseClaim referencing entities that exist in GMS
tool_claim = ToolUseClaim(
    tool_name="MDL-001",
    tool_args={"check": "risk"},
    result_key="High",
    claimed_value="High",
    confidence=1.0,
    provenance=ClaimProvenance(extraction_method="metadata"),
)

sv_tool = policy_v.verify(tool_claim)
print("=== ToolUseClaim (entities in GMS) ===")
print(f"  Tool:     {tool_claim.tool_name}")
print(f"  Result:   {tool_claim.result_key}")
print(f"  Passed:   {sv_tool.passed}")
print(f"  Confidence: {sv_tool.confidence:.4f}")
print(f"  GMS method: {sv_tool.gms_method_used}")
print(f"  Detail:   {sv_tool.detail[:120]}")
if sv_tool.failure:
    print(f"  Failure:  {sv_tool.failure.code.value}")

=== ToolUseClaim (entities in GMS) ===
  Tool:     MDL-001
  Result:   High
  Passed:   True
  Confidence: 0.8000
  GMS method: query_triples
  Detail:   (MDL-001, *, High) found in graph: ['has_risk_rating']


In [16]:
# ToolUseClaim with completely unknown tool (not in GMS)
tool_unknown = ToolUseClaim(
    tool_name="unknown_api_v9",
    tool_args={},
    result_key="mystery_result",
    claimed_value="42",
    confidence=1.0,
    provenance=ClaimProvenance(extraction_method="metadata"),
)

sv_tunknown = policy_v.verify(tool_unknown)
print("=== ToolUseClaim (unknown tool) ===")
print(f"  Tool:     {tool_unknown.tool_name}")
print(f"  Result:   {tool_unknown.result_key}")
print(f"  Passed:   {sv_tunknown.passed}")
print(f"  GMS method: {sv_tunknown.gms_method_used}")
print(f"  Detail:   {sv_tunknown.detail}")
if sv_tunknown.failure:
    print(f"  Failure:  {sv_tunknown.failure.code.value}")
    print(f"  Action:   {sv_tunknown.failure.suggested_action}")

=== ToolUseClaim (unknown tool) ===
  Tool:     unknown_api_v9
  Result:   mystery_result
  Passed:   False
  GMS method: fuzzy_match_entity
  Detail:   Tool 'unknown_api_v9' and result 'mystery_result' not in GMS — cannot verify
  Failure:  policy_violation
  Action:   Register tool entities in the knowledge graph during ingestion


---

## The VerificationRouter

Individual verifiers are useful for understanding the mechanics, but in practice the `VerificationRouter` handles dispatch automatically. It maintains a dispatch table mapping `claim_type` strings to verifier methods:

```python
self._dispatch = {
    "fact":      self.graph.verify,
    "threshold": self.numeric.verify,
    "artifact":  self.artifact.verify,
    "rule":      self.logic.verify,
    "tool_use":  self.policy.verify,
}
```

The `route()` method looks up the handler and returns a SubVerdict. The `route_all()` method processes a list of claims.

In [17]:
# Lab 9.2a: Create the VerificationRouter wired to the calibrated judge.
# The router reads judge._thresholds and injects geodesic / holonomy cuts
# into GraphVerifier and LogicVerifier; verifiers without a calibration
# corpus keep their hand-picked defaults.
router = VerificationRouter(store, judge=judge)
print("Router verifier thresholds:")
print(f"  GraphVerifier.plausibility_threshold = {router.graph.plausibility_threshold:.4f}")
print(f"  LogicVerifier.config.tau_path        = {router.logic.config.tau_path:.4f}")
print(f"  PolicyVerifier.plausibility_threshold = {router.policy.plausibility_threshold:.4f} (default)")

# Verify a mixed set of claims
claims = [fact_correct, fact_wrong, fact_missing,
          threshold_pass, artifact_correct, artifact_wrong]

verdicts = router.route_all(claims)

print(f"\nRouted {len(claims)} claims through VerificationRouter:\n")
print(f"{'#':<3s} {'Type':<12s} {'Passed':>7s} {'Conf':>6s} {'GMS Method':<20s} {'Failure Code'}")
print("-" * 75)
for i, (claim, sv) in enumerate(zip(claims, verdicts), 1):
    fc = sv.failure.code.value if sv.failure else "—"
    print(f"{i:<3d} {claim.claim_type:<12s} {str(sv.passed):>7s} "
          f"{sv.confidence:>6.2f} {sv.gms_method_used:<20s} {fc}")

Router verifier thresholds:
  GraphVerifier.plausibility_threshold = 1.3883
  LogicVerifier.config.tau_path        = 2.3492
  PolicyVerifier.plausibility_threshold = 1.5000 (default)

Routed 6 claims through VerificationRouter:

#   Type          Passed   Conf GMS Method           Failure Code
---------------------------------------------------------------------------
1   fact            True   0.27 score_triple         —
2   fact           False   0.22 score_triple         fact_incorrect
3   fact           False   0.00 fuzzy_match_entity   entity_not_found
4   threshold      False   0.00 check_threshold      verification_error
5   artifact        True   1.00 enm.verify_integrity —
6   artifact       False   0.00 enm.verify_integrity exact_mismatch


### Routing Claims from the ClaimCompiler

The real power emerges when we connect the ClaimCompiler (Chapter 8) to the VerificationRouter. Generate DOE questions, compile claims and verify them all.

In [18]:
# Lab 9.2b: End-to-end — DOE questions -> claims -> verification
doe_questions = bench.generate_questions()
compiler = ClaimCompiler(store=store)


def make_answer(q):
    """Simulate a correct agent answer from question metadata.

    The rule-based extractor needs explicit mention of head/tail/chain
    entities in the answer text to recover claim content. Concatenating
    the ground-truth entities mirrors an agent that names the expected
    entities in its reply.
    """
    m = q.metadata
    parts = []
    for k in (
        "head", "tail", "true_tail", "mid", "mid1", "mid2",
        "value", "entity_id", "entity_a", "entity_b",
    ):
        v = m.get(k)
        if v is not None:
            parts.append(str(v))
    chain = m.get("chain")
    if isinstance(chain, list):
        parts.extend(str(x) for x in chain)
    return " ".join(parts) if parts else "Unknown"


# Compile and verify claims from first 40 DOE questions (one per category)
all_claims = []
all_verdicts = []

for i, dq in enumerate(doe_questions[:40]):
    q = dq.question
    agent_answer = make_answer(q)

    claims = compiler.compile(
        llm_answer=agent_answer,
        question_metadata=q.metadata,
        question_category=q.category,
        run_id="ch09-demo",
        question_id=f"q-{i:03d}",
        ground_truth=str(q.ground_truth),
        question_text=q.natural_language,
    )
    verdicts = router.route_all(claims)
    all_claims.extend(claims)
    all_verdicts.extend(verdicts)

print(f"Compiled {len(all_claims)} claims from 40 DOE questions")
print(f"Verified {len(all_verdicts)} claims via VerificationRouter\n")

# Pass/fail summary
passed = sum(1 for sv in all_verdicts if sv.passed)
failed = len(all_verdicts) - passed
print(f"Passed: {passed} ({passed/len(all_verdicts)*100:.1f}%)")
print(f"Failed: {failed} ({failed/len(all_verdicts)*100:.1f}%)")

  Base questions: 130 (13 generators, max 10/cat)

  DOE expansion: 130 base x 4 runs = 520 questions
  Factors: ['clarity', 'entity_aliasing', 'reasoning_cue']
  [BackgroundTrain] Starting: 1 entities, 0 relations, 22 triples


  [BackgroundTrain] Done: Trained: 1 entities, 0 relations, 50 steps in 3.6s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved


Compiled 92 claims from 40 DOE questions
Verified 92 claims via VerificationRouter

Passed: 79 (85.9%)
Failed: 13 (14.1%)


In [19]:
# Verdicts by claim type
type_pass = Counter()
type_fail = Counter()
type_total = Counter()

for claim, sv in zip(all_claims, all_verdicts):
    type_total[claim.claim_type] += 1
    if sv.passed:
        type_pass[claim.claim_type] += 1
    else:
        type_fail[claim.claim_type] += 1

print(f"\n{'Claim Type':<14s} {'Total':>6s} {'Pass':>6s} {'Fail':>6s} {'Pass%':>7s}")
print("-" * 43)
for ctype in sorted(type_total.keys()):
    total = type_total[ctype]
    p = type_pass[ctype]
    f = type_fail[ctype]
    pct = p / total * 100 if total > 0 else 0
    print(f"{ctype:<14s} {total:>6d} {p:>6d} {f:>6d} {pct:>6.1f}%")


Claim Type      Total   Pass   Fail   Pass%
-------------------------------------------
artifact           25     25      0  100.0%
fact               47     41      6   87.2%
rule               20     13      7   65.0%


In [20]:
# GMS methods used across all verdicts
method_counts = Counter(sv.gms_method_used for sv in all_verdicts)

print(f"\n{'GMS Method':<22s} {'Count':>7s} {'Pct':>7s}")
print("-" * 40)
for method, count in method_counts.most_common():
    pct = count / len(all_verdicts) * 100
    print(f"{method:<22s} {count:>7d} {pct:>6.1f}%")


GMS Method               Count     Pct
----------------------------------------
score_triple                45   48.9%
enm.verify_integrity        25   27.2%
check_holonomy               9    9.8%
none                         7    7.6%
tension_energy               6    6.5%


In [21]:
# Failure code distribution (for failed claims)
failure_codes = Counter(
    sv.failure.code.value for sv in all_verdicts if sv.failure
)

if failure_codes:
    print(f"\n{'Failure Code':<30s} {'Count':>7s}")
    print("-" * 40)
    for code, count in failure_codes.most_common():
        print(f"{code:<30s} {count:>7d}")
else:
    print("\nNo failures detected (all claims passed)")


Failure Code                     Count
----------------------------------------
incomplete_answer                    7
fact_incorrect                       4
holonomy_defect                      2


In [22]:
# Show a few detailed SubVerdicts (mix of pass and fail)
print("\n=== Sample SubVerdicts (detailed) ===")
shown_pass, shown_fail = 0, 0
for claim, sv in zip(all_claims, all_verdicts):
    if sv.passed and shown_pass < 2:
        shown_pass += 1
    elif not sv.passed and shown_fail < 2:
        shown_fail += 1
    else:
        continue

    print(f"\n  [{claim.claim_type}] claim_id={claim.claim_id}")
    if claim.claim_type == "fact":
        print(f"    triple: ({claim.head}, {claim.relation}, {claim.tail})")
    elif claim.claim_type == "artifact":
        print(f"    {claim.category}({claim.entity_id}) = {claim.claimed_value}")
    elif claim.claim_type == "rule":
        print(f"    path: {' -> '.join(claim.antecedent_entities)}")
    print(f"    passed={sv.passed}, confidence={sv.confidence:.4f}")
    print(f"    gms_method={sv.gms_method_used}")
    print(f"    detail: {sv.detail[:100]}")
    if sv.failure:
        print(f"    FAILURE: {sv.failure.code.value}")
        print(f"    gms_method: {sv.failure.gms_method}")

    if shown_pass >= 2 and shown_fail >= 2:
        break


=== Sample SubVerdicts (detailed) ===

  [fact] claim_id=5fefaea900d0
    triple: (Chief Model Risk Officer, has_detail, Dr. Elena Vasquez)
    passed=True, confidence=0.3480
    gms_method=score_triple
    detail: (Chief Model Risk Officer, has_detail, Dr. Elena Vasquez) geodesic=1.0556

  [fact] claim_id=66a30f92061c
    triple: (Total Models in Inventory, in_section, executive_summary)
    passed=True, confidence=0.3637
    gms_method=score_triple
    detail: (Total Models in Inventory, in_section, executive_summary) geodesic=1.0114

  [fact] claim_id=10f231ce332d
    triple: (Medium-Risk Models, has_detail, )
    passed=False, confidence=0.0000
    gms_method=none
    detail: Agent did not assert a tail for (Medium-Risk Models, has_detail, ?)
    FAILURE: incomplete_answer
    gms_method: none

  [rule] claim_id=6341ccd5e7c9
    path: 
    passed=False, confidence=0.0000
    gms_method=none
    detail: Agent did not assert the rule
    FAILURE: incomplete_answer
    gms_method: no

### Full 520-Question Verification

Let us scale to all 520 DOE questions and examine the verification profile across the full benchmark.

In [23]:
# Full 520-question verification — reuses the make_answer simulator above
full_claims = []
full_verdicts = []

for i, dq in enumerate(doe_questions):
    q = dq.question
    agent_answer = make_answer(q)

    claims = compiler.compile(
        llm_answer=agent_answer,
        question_metadata=q.metadata,
        question_category=q.category,
        run_id="ch09-full",
        question_id=f"q-{i:03d}",
        ground_truth=str(q.ground_truth),
        question_text=q.natural_language,
    )
    verdicts = router.route_all(claims)
    full_claims.extend(claims)
    full_verdicts.extend(verdicts)

full_passed = sum(1 for sv in full_verdicts if sv.passed)
full_failed = len(full_verdicts) - full_passed

print(f"Full benchmark verification ({len(doe_questions)} questions):")
print(f"  Total claims:  {len(full_claims)}")
print(f"  Total verdicts: {len(full_verdicts)}")
print(f"  Passed: {full_passed} ({full_passed/len(full_verdicts)*100:.1f}%)")
print(f"  Failed: {full_failed} ({full_failed/len(full_verdicts)*100:.1f}%)")

Full benchmark verification (520 questions):
  Total claims:  1496
  Total verdicts: 1496
  Passed: 1232 (82.4%)
  Failed: 264 (17.6%)


In [24]:
# Full results by claim type
ft_pass = Counter()
ft_fail = Counter()
ft_total = Counter()
ft_methods = Counter()

for claim, sv in zip(full_claims, full_verdicts):
    ft_total[claim.claim_type] += 1
    ft_methods[sv.gms_method_used] += 1
    if sv.passed:
        ft_pass[claim.claim_type] += 1
    else:
        ft_fail[claim.claim_type] += 1

print("\nVerdicts by claim type:")
print(f"{'Claim Type':<14s} {'Total':>6s} {'Pass':>6s} {'Fail':>6s} {'Pass%':>7s}")
print("-" * 43)
for ctype in sorted(ft_total.keys()):
    total = ft_total[ctype]
    p = ft_pass[ctype]
    f_ = ft_fail[ctype]
    pct = p / total * 100 if total > 0 else 0
    print(f"{ctype:<14s} {total:>6d} {p:>6d} {f_:>6d} {pct:>6.1f}%")

# GMS methods
print("\nGMS methods used:")
print(f"{'Method':<22s} {'Count':>7s} {'Pct':>7s}")
print("-" * 40)
for method, count in ft_methods.most_common():
    pct = count / len(full_verdicts) * 100
    print(f"{method:<22s} {count:>7d} {pct:>6.1f}%")

# Failure codes
full_fcodes = Counter(
    sv.failure.code.value for sv in full_verdicts if sv.failure
)
if full_fcodes:
    print("\nFailure codes:")
    print(f"{'Code':<30s} {'Count':>7s}")
    print("-" * 40)
    for code, count in full_fcodes.most_common():
        print(f"{code:<30s} {count:>7d}")


Verdicts by claim type:
Claim Type      Total   Pass   Fail   Pass%
-------------------------------------------
artifact          272    272      0  100.0%
fact             1084    884    200   81.5%
process            20      0     20    0.0%
rule              120     76     44   63.3%

GMS methods used:
Method                   Count     Pct
----------------------------------------
score_triple               944   63.1%
enm.verify_integrity       272   18.2%
none                        96    6.4%
query_triples               76    5.1%
tension_energy              48    3.2%
check_holonomy              36    2.4%
routing                     20    1.3%
fuzzy_match_entity           4    0.3%

Failure codes:
Code                             Count
----------------------------------------
fact_incorrect                     132
incomplete_answer                   96
verification_error                  20
holonomy_defect                      8
relation_not_found                   4
entity_no

In [25]:
# Confidence distribution of passing verdicts
pass_confs = [sv.confidence for sv in full_verdicts if sv.passed]

if pass_confs:
    print("\nConfidence distribution (passing verdicts):")
    print(f"  Count:  {len(pass_confs)}")
    print(f"  Mean:   {np.mean(pass_confs):.4f}")
    print(f"  Median: {np.median(pass_confs):.4f}")
    print(f"  Min:    {np.min(pass_confs):.4f}")
    print(f"  Max:    {np.max(pass_confs):.4f}")

    # Histogram buckets
    buckets = [(0, 0.5), (0.5, 0.7), (0.7, 0.9), (0.9, 1.01)]
    print(f"\n  {'Confidence Range':<20s} {'Count':>7s} {'Pct':>7s}")
    print(f"  {'-'*38}")
    for lo, hi in buckets:
        n = sum(1 for c in pass_confs if lo <= c < hi)
        pct = n / len(pass_confs) * 100
        print(f"  [{lo:.1f}, {hi:.1f}){' '*(14-len(f'[{lo:.1f}, {hi:.1f})'))}"
              f" {n:>7d} {pct:>6.1f}%")


Confidence distribution (passing verdicts):
  Count:  1232
  Mean:   0.4559
  Median: 0.3045
  Min:    0.2265
  Max:    1.0000

  Confidence Range       Count     Pct
  --------------------------------------
  [0.0, 0.5)         960   77.9%
  [0.5, 0.7)           0    0.0%
  [0.7, 0.9)           0    0.0%
  [0.9, 1.0)         272   22.1%


---

## Exercises

### Exercise 9.1

Construct a `FactClaim` where the head and tail entities exist in the store but are connected by a different relation than the one claimed. Verify it with `GraphVerifier`. What failure code do you get? Compare the geodesic distance to the plausibility threshold (1.2).

### Exercise 9.2

Run the full 520-question verification but use an **incorrect** answer for every question (e.g., "Unknown"). How does the failure code distribution change compared to the correct-answer run? Which claim types are most affected?

### Exercise 9.3

The `ArtifactVerifier` uses SHA-256 hash comparison rather than floating-point tolerance. Explain why this design choice matters for regulated environments. Construct two `ArtifactClaim`s where the claimed values differ by less than $10^{-10}$ — does the verifier catch the difference?

In [26]:
# === Exercise 9.1: Your code here ===
# Hint:
# fact_wrong_rel = FactClaim(
#     head="MDL-001", relation="has_status", tail="High",  # High is risk, not status
#     confidence=1.0,
# )
# sv = graph_v.verify(fact_wrong_rel)
# print(sv.failure.code, sv.raw_score)

In [27]:
# === Exercise 9.2: Your code here ===
# Hint: reuse the full-benchmark loop but set agent_answer = "Unknown"

In [28]:
# === Exercise 9.3: Your code here ===
# Hint:
# a1 = ArtifactClaim(category=..., entity_id=..., claimed_value=0.8412)
# a2 = ArtifactClaim(category=..., entity_id=..., claimed_value=0.8412 + 1e-11)
# Both should use the same ENM key. Does SHA-256 catch 1e-11?

---

## Summary

This chapter introduced the verifier layer — the bridge between typed claims and GMS geometric primitives:

1. **Five structural verifiers** (Graph, Numeric, Artifact, Logic, Policy) each call specific GMS methods and produce `SubVerdict` objects with pass/fail, confidence, raw geometric signal and failure attribution.
2. **GraphVerifier** uses a three-step cascade: neural `score_triple()` -> graph `query_triples()` -> pair existence. Confidence = $e^{-\text{geodesic}}$.
3. **NumericVerifier** uses phase-encoded threshold checks — no Python arithmetic. If no phase encoder exists, it fails.
4. **ArtifactVerifier** uses SHA-256 hash comparison — deterministic, tolerance-free verification of exact numeric values.
5. **LogicVerifier** checks path consistency via holonomy defect and contradiction via tension energy.
6. **PolicyVerifier** checks tool-use plausibility via geodesic scoring on tool-result triples.
7. **VerificationRouter** dispatches claims automatically via a `claim_type -> verifier` dispatch table, with error handling for unroutable claims.
8. **FailureAttribution** links every failure to a specific GMS method and failure code, enabling systematic root-cause analysis.

**Next:** Chapter 10 aggregates SubVerdicts into **TypedVerdicts**, introduces the failure code taxonomy and classifies failure severity with risk-tier awareness.